# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [45]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [46]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    B,C,H,W = x.shape

    assert H % patch_size == 0 and W % patch_size == 0

    x = x.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    x = x.permute(0, 2, 3, 1, 4, 5)
    x = x.reshape(B, -1, C * patch_size * patch_size)
    return x


In [47]:
# TODO: Add positional encoding as done in the ViT paper and patch projection
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        #patch dim = patch_size * patch_size, d_model
        super().__init__()
        self.patch_dim = patch_dim
        self.d_model = d_model
        self.lin_proj = nn.Linear(self.patch_dim, self.d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        return self.lin_proj(x_patches)


class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, num_tokens, d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pos_embed

In [48]:
# TODO: Define the variants you want to compare against each other from the GLU paper. Justify your choice.
class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.FFN = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model, bias=False),
            nn.Dropout(dropout),
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.FFN(x)



class GLUFeedForward(nn.Module):
    """GLU-family FFN""" 
    VALID_VARIANTS = {"GEGLU", "SwiGLU"}
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        if variant not in self.VALID_VARIANTS:
            raise ValueError(
                f"Invalid GLU variant '{variant}'. "
                f"Expected one of {sorted(self.VALID_VARIANTS)}."
            )

        self.d_model = d_model
        self.d_ff_gated = d_ff_gated
        self.variant = variant

        self.gate_proj = nn.Linear(d_model, d_ff_gated, bias=False)
        self.value_proj = nn.Linear(d_model, d_ff_gated, bias=False)
        self.out_proj = nn.Linear(d_ff_gated, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    def _activation(self, x: torch.Tensor) -> torch.Tensor:
        if self.variant == "GEGLU":
            return F.gelu(x)
        if self.variant == "SwiGLU":
            return F.silu(x)
        
        raise RuntimeError(f"Unexpected variant: {self.variant}")
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = self.gate_proj(x)
        value = self.value_proj(x)
        
        hidden = self._activation(gate) * value
        hidden = self.dropout(hidden)

        out_proj = self.out_proj(hidden)

        return out_proj


In [49]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        self.mlp = mlp
        self.attention = nn.MultiheadAttention(
            embed_dim = d_model,
            num_heads = n_heads,
            dropout = dropout,
            batch_first = True)
        self.dropout = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.attention(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + self.dropout(attn_out)

        mlp_out = self.mlp(self.ln2(x))
        x = x + self.dropout(mlp_out)
        return x


In [50]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size

        d_ff_gated = d_ff * 2 // 3

        valid_variations = ("GEGLU", "SwiGLU")

        self.embeddings = PatchEmbed(patch_dim, d_model)
        self.pos_embeddings = PositionalEmbedding(self.num_tokens, d_model)
        
        if mlp_kind == "GELU":
            self.mlp = FeedForward(d_model, d_ff, dropout)
        elif mlp_kind in valid_variations:
            self.mlp = GLUFeedForward(d_model, d_ff_gated, dropout, mlp_kind)
        else:
            raise ValueError(
                f"Invalid GLU variant '{mlp_kind}'. "
                f"Expected one of {sorted(valid_variations) + ['GELU']}"
            )
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=self.mlp,
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        self.num_classes = 10
        self.head = nn.Linear(d_model, self.num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        patches = patchify(x, self.patch_size)
        embeddings = self.embeddings(patches)
        pos_embeddings = self.pos_embeddings(embeddings)

        for block in self.blocks:
            pos_embeddings = block(pos_embeddings)

        pooled = pos_embeddings.mean(dim=1)
        logits = self.head(pooled)
        return logits

In [51]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cuda"  # set "cuda" if available

In [52]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    test_accs: list[float] = []

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        epoch_losses = []
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = nn.CrossEntropyLoss()(logits, yb) 

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())
            epoch_losses.append(loss.item())

        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        avg_train_loss = sum(epoch_losses) / len(epoch_losses)
        final_epoch_loss = epoch_losses[-1]
        print(
            f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | "
            f"epoch avg train loss: {avg_train_loss:.4f} | "
            f"epoch final batch loss: {final_epoch_loss:.4f} | "
            f"test acc: {test_accs[-1]:.4f}"
        )

    res = {
        "final_acc": test_accs[-1],
        "best_acc": max(test_accs),
        "final_train_loss": train_losses[-1],
        "final_epoch_train_loss": avg_train_loss,
        "final_epoch_batch_loss": final_epoch_loss,
        "train_losses": train_losses,
        "test_accs": test_accs,
    }
    return res

In [ ]:
cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01, device="cpu")
tfm = transforms.Compose([transforms.ToTensor()])

g = torch.Generator()
g.manual_seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, generator=g, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, generator=g, num_workers=0)

# Tiny model example. TODO: You're welcome to experiment with these parameters
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 4 * d_model
dropout = 0.1

runs = ["GELU", "GEGLU", "SwiGLU"] # TODO: Name your runs
results = []

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for kind in runs:
    model = TinyViT(
        patch_size=patch_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        dropout=dropout,
        mlp_kind=kind,
    )
    num_params = count_parameters(model)
    print(f"\n{'='*60}")
    print(f"Run: {kind} | Parameters: {num_params:,}")
    print(f"{'='*60}")
    out = train_one_run(kind, model, train_loader, test_loader, cfg)
    out["num_params"] = num_params
    out["mlp_kind"] = kind
    results.append(out)

print(f"\n{'='*60}")
print("RESULTS SUMMARY")
print(f"{'='*60}")
for res in results:
    kind = res["mlp_kind"]
    print(f"\n{kind}:")
    print(f"  Parameters:              {res['num_params']:,}")
    print(f"  Final Acc:               {res['final_acc']:.4f}")
    print(f"  Best Acc:                {res['best_acc']:.4f}")
    print(f"  Final batch train loss:  {res['final_train_loss']:.4f}")
    print(f"  Final epoch avg loss:    {res['final_epoch_train_loss']:.4f}")


Run: GELU | Parameters: 71,434
